# RSIC Volatility Forecasting Audit (v7): Loss-Matched Retraining & 60-Day Periodic Refitting
**Research Question 2 Audit:** Resolving loss-architecture confounding and refitting asymmetry in deep learning volatility forecasting of BTC/IDR Indodax ($r_t^2$).

### Experimental Protocols:
1. **Stage A (Untuned Loss-Matched Retraining):** TFT with MAE loss, N-BEATSx with MSE loss using original v6 winning hyperparameters ($N=335$ cutoffs, 5 seeds). Preserved for exact reproducibility of v7-a results.
2. **Stage A_TUNED (Properly Tuned Loss-Matched Retraining):** N-BEATSx with MSE loss tuned across an 8-combo validation grid ($N_{\text{val}}=167$ rows, zero test leakage). The winning hyperparameter configuration is evaluated across 5 seeds on the full test set (Config: `NBEATSx_MSEtuned_SPILL` and `NBEATSx_MSEtuned_NOSPILL`).
3. **Stage B (60-Day Periodic Expanding Refit):** 6 expanding blocks of 60 days across the 335 test cutoffs (Seed 42), retraining every 60 days with strict anti-leakage boundaries.
4. **Stage ALL:** Executes Stage A_TUNED first, saves all outputs, and then executes Stage B with a total runtime budget guard (`MAX_RUNTIME_HOURS`).


# CELL 1: ENVIRONMENT SETUP, PINNING NEURALFORECAST 3.3.0, GLOBAL SEEDS & CONTROLS


In [1]:
import os
import sys
import subprocess
import random
import hashlib
import json
import time
import glob
import copy
import shutil
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# --------------------------------------------------------------------------------------
# RUN CONTROLS: Choose STAGE = 'SMOKE', 'A', 'A_TUNED', 'B', or 'ALL'
# --------------------------------------------------------------------------------------
STAGE = "ALL"  # Options: 'SMOKE', 'A', 'A_TUNED', 'B', 'ALL' (Default: 'SMOKE')
MAX_RUNTIME_HOURS = 10.0  # Maximum execution time budget in hours (guard against timeout)
RUN_TFT_MAE_TUNED = True  # Set True to tune TFT with MAE loss alongside NBEATSx with MSE loss

# Global tracking structures for audit provenance and stage timers
STAGE_TIMINGS = {
    'A': None,
    'A_TUNED': None,
    'B': None,
    'POSTPROCESSING': None
}
MISSING_BASELINES = []
PARTIAL_ENSEMBLES = []

T_GLOBAL_START = time.time()

# Check Python & Internet Connectivity
print(f'Python Version   : {sys.version.split()[0]}')
has_net = False
try:
    import socket
    socket.setdefaulttimeout(3)
    socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect(('8.8.8.8', 53))
    has_net = True
except Exception:
    has_net = False
print(f'Status Internet  : {"ONLINE" if has_net else "OFFLINE"}')

# Pinning neuralforecast==3.3.0 (same version as v6)
try:
    import neuralforecast
    nf_ver = neuralforecast.__version__
except ImportError:
    nf_ver = None

if has_net and (os.path.exists('/kaggle') or (nf_ver != '3.3.0') or (nf_ver is None)):
    print('Menginstal/menyelaraskan paket neuralforecast==3.3.0, arch, statsmodels...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'neuralforecast==3.3.0', 'arch', 'statsmodels'], check=True)
    import neuralforecast
    nf_ver = neuralforecast.__version__

import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, Callback

print(f'PyTorch Version  : {torch.__version__}')
print(f'Lightning Version: {pl.__version__}')
print(f'NeuralForecast   : {nf_ver}')

# Assert version pinning
if nf_ver is not None:
    assert nf_ver == '3.3.0', f'ASSERT STOP: neuralforecast harus versi 3.3.0, ditemukan {nf_ver}!'
else:
    print('WARNING: neuralforecast not installed in this environment.')

# Device Audit
gpu_available = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if gpu_available else 'CPU'
print(f'Hardware Device  : {gpu_name} (CUDA={gpu_available})')

# Deterministic Seed Helper
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    pl.seed_everything(seed, workers=True)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_all_seeds(42)

# Execution banner parameters
N_TEST_CUTOFFS = 335 if STAGE != 'SMOKE' else 3
MAX_STEPS_BANNER = 1000 if STAGE != 'SMOKE' else 20
VAL_CHECK_STEPS_BANNER = 25 if STAGE != 'SMOKE' else 10
PATIENCE_BANNER = 5 if STAGE != 'SMOKE' else 2
SEEDS_BANNER = [42, 7, 13, 21, 99] if STAGE != 'SMOKE' else [42]

print('=' * 88)
print(f'>>> ACTIVE EXECUTION STAGE: [{STAGE}] | BUDGET: {MAX_RUNTIME_HOURS}h <<<')
print(f'>>> MAX_STEPS: {MAX_STEPS_BANNER} | VAL_CHECK_STEPS: {VAL_CHECK_STEPS_BANNER} | PATIENCE: {PATIENCE_BANNER} checks <<<')
print(f'>>> SEEDS: {SEEDS_BANNER} | TEST CUTOFFS: {N_TEST_CUTOFFS} | NEURALFORECAST: {nf_ver} <<<')
if STAGE == 'SMOKE':
    print('NOTICE: SMOKE mode will execute 1 seed, 3 cutoffs to verify pipeline & estimate T4 GPU runtime.')
elif STAGE == 'ALL':
    print('NOTICE: STAGE ALL will execute Stage A_TUNED first, followed by Stage B.')
print('=' * 88)


Python Version   : 3.13.15
Status Internet  : ONLINE
Menginstal/menyelaraskan paket neuralforecast==3.3.0, arch, statsmodels...
PyTorch Version  : 2.11.0+cu128
Lightning Version: 2.6.6
NeuralForecast   : 3.3.0


Seed set to 42


Hardware Device  : Tesla T4 (CUDA=True)
>>> ACTIVE EXECUTION STAGE: [ALL] | BUDGET: 10.0h <<<
>>> MAX_STEPS: 1000 | VAL_CHECK_STEPS: 25 | PATIENCE: 5 checks <<<
>>> SEEDS: [42, 7, 13, 21, 99] | TEST CUTOFFS: 335 | NEURALFORECAST: 3.3.0 <<<
NOTICE: STAGE ALL will execute Stage A_TUNED first, followed by Stage B.


# CELL 2: KONFIGURASI GLOBAL, HYPERPARAMETERS DARI V6, FITUR & DIREKTORI


In [2]:

# 1. Feature sets
NOSPILL_COLS = [
    'ret_saham_nvda', 'ret_saham_msft', 'ret_saham_tsla', 'ret_saham_googl', 'ret_saham_meta',
    'ret_crypto_btc_usd', 'ret_crypto_eth_usd', 'crypto_btc_usd_weekend_return',
    'gap_days', 'is_monday', 'is_ihsg_holiday'
]

SPILL_COLS = NOSPILL_COLS + [
    'spillover_ai_to_btcidr', 'tvp_tsi',
    'net_spill_saham_nvda', 'net_spill_saham_msft', 'net_spill_saham_tsla', 'net_spill_saham_googl', 'net_spill_saham_meta',
    'net_spill_crypto_btc_usd', 'net_spill_crypto_eth_usd'
]

FEATURE_SETS = {
    'NOSPILL': NOSPILL_COLS,
    'SPILL': SPILL_COLS
}

SEEDS = SEEDS_BANNER
H = 10

# 2. Hyperparameters from results/v6/v6_best_hyperparameters.json
V6_BEST_HP = {
    'TFT': {
        'input_size': 50,
        'learning_rate': 0.001,
        'hidden_size': 64,
        'max_steps': MAX_STEPS_BANNER,
        'val_check_steps': VAL_CHECK_STEPS_BANNER,
        'patience': PATIENCE_BANNER
    },
    'NBEATSx': {
        'input_size': 20,
        'learning_rate': 0.0003,
        'mlp_units': [[256, 256], [256, 256], [256, 256]],
        'max_steps': MAX_STEPS_BANNER,
        'val_check_steps': VAL_CHECK_STEPS_BANNER,
        'patience': PATIENCE_BANNER
    }
}

# 3. Directory paths
BASE_DIR = os.path.abspath('..') if os.path.basename(os.getcwd()) in ['notebooks', 'followup'] else os.path.abspath('.')
KAGGLE_INPUT_DIR = '/kaggle/input/datasets/ferastoro/ai-stocks-crypto-and-ihsg/data_clean'
LOCAL_INPUT_DIR = os.path.join(BASE_DIR, 'data', 'clean')
FALLBACK_INPUT_DIR = os.path.join(BASE_DIR, 'inputs', 'v5')

if os.path.exists(KAGGLE_INPUT_DIR):
    DATA_DIR = KAGGLE_INPUT_DIR
elif os.path.exists(LOCAL_INPUT_DIR):
    DATA_DIR = LOCAL_INPUT_DIR
else:
    DATA_DIR = FALLBACK_INPUT_DIR

OUTPUT_DIR = '/kaggle/working' if os.path.exists('/kaggle') else os.path.join(BASE_DIR, 'audit_outputs', 'followup', 'v7_results')
LOGS_DIR = os.path.join(OUTPUT_DIR, 'logs')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

# Central Run Log file
RUN_LOG_CSV = os.path.join(OUTPUT_DIR, 'run_log.csv')

def atomic_save_csv(df: pd.DataFrame, target_path: str):
    """Write DataFrame atomically using a temporary file to avoid incomplete files."""
    tmp_path = f"{target_path}.tmp_{os.getpid()}_{random.randint(1000, 9999)}"
    df.to_csv(tmp_path, index=False)
    os.replace(tmp_path, target_path)

def log_run_metrics(metrics: dict):
    """Append one run entry atomically to the central run_log.csv."""
    df_new = pd.DataFrame([metrics])
    if os.path.exists(RUN_LOG_CSV):
        try:
            df_existing = pd.read_csv(RUN_LOG_CSV)
            df_combined = pd.concat([df_existing[df_existing['run_id'] != metrics.get('run_id')], df_new], ignore_index=True)
        except Exception:
            df_combined = df_new
    else:
        df_combined = df_new
    atomic_save_csv(df_combined, RUN_LOG_CSV)

def check_runtime_budget(estimated_next_seconds: float = 0.0) -> bool:
    """Return False if remaining time budget is insufficient for the next fit."""
    elapsed = time.time() - T_GLOBAL_START
    budget_seconds = MAX_RUNTIME_HOURS * 3600.0
    reserve_seconds = 600.0  # 10 minutes reserve for packaging and postprocessing
    if elapsed + estimated_next_seconds + reserve_seconds > budget_seconds:
        print(f"\n⚠️ RUNTIME BUDGET GUARD: Elapsed {elapsed/3600:.2f}h + estimated {estimated_next_seconds/60:.1f}m + reserve 10m exceeds budget {MAX_RUNTIME_HOURS}h!")
        return False
    return True

print(f'Input Data Directory : {DATA_DIR}')
print(f'Output Directory     : {OUTPUT_DIR}')
print(f'Logs Directory       : {LOGS_DIR}')
print(f'Run Log File         : {RUN_LOG_CSV}')
print(f'Test Cutoffs Target  : {N_TEST_CUTOFFS} cutoffs')


Input Data Directory : /kaggle/input/datasets/ferastoro/ai-stocks-crypto-and-ihsg/data_clean
Output Directory     : /kaggle/working
Logs Directory       : /kaggle/working/logs
Run Log File         : /kaggle/working/run_log.csv
Test Cutoffs Target  : 335 cutoffs


# CELL 3: DATA LOADING, AUTO-DISCOVERY /KAGGLE/INPUT, MD5, SCHEMA, SPLIT 1165/167/344


In [3]:
# ======================================================================================
# CELL 3: DATA LOADING, AUTO-DISCOVERY /KAGGLE/INPUT, MD5, SCHEMA, SPLIT 1165/167/344
# ======================================================================================
import glob

def find_file(filename, candidate_dirs=None):
    if candidate_dirs:
        for d in candidate_dirs:
            if d and os.path.exists(d):
                p = os.path.join(d, filename)
                if os.path.exists(p):
                    return os.path.abspath(p)
    if os.path.exists('/kaggle/input'):
        matches = glob.glob(f'/kaggle/input/**/{filename}', recursive=True)
        if matches:
            return os.path.abspath(matches[0])
    for base in ['.', '..', '../..']:
        for sub in ['inputs/v5', 'data/clean', 'scratch/final', 'rsic-v5-inputs', 'results/v6', 'v6_outputs', 'results']:
            p = os.path.join(base, sub, filename)
            if os.path.exists(p):
                return os.path.abspath(p)
        matches = glob.glob(f'{base}/**/{filename}', recursive=True)
        if matches:
            return os.path.abspath(matches[0])
    raise FileNotFoundError(f'File {filename} tidak ditemukan di /kaggle/input maupun folder lokal!')

def get_file_md5(p):
    if not os.path.exists(p):
        return 'NOT_FOUND'
    h = hashlib.md5()
    with open(p, 'rb') as fp:
        while chunk := fp.read(8192):
            h.update(chunk)
    return h.hexdigest()

dl_file = find_file('dataset_deep_learning.csv', [DATA_DIR, FALLBACK_INPUT_DIR])
tvp_file = find_file('tvp_var_dynamic_features_causal.csv', [DATA_DIR, FALLBACK_INPUT_DIR])

print('=== HASIL PENEMUAN FILE DATASET (AUTO-DISCOVERY) ===')
print(f'dataset_deep_learning.csv            : {dl_file}')
print(f'tvp_var_dynamic_features_causal.csv  : {tvp_file}')

df_dl = pd.read_csv(dl_file)
df_tvp = pd.read_csv(tvp_file)

df_dl['Date'] = pd.to_datetime(df_dl['Date'])
df_tvp['Date'] = pd.to_datetime(df_tvp['Date'])

# Target column detection (v6 standard: ret_crypto_btc_idr_indodax)
target_ret_col = 'ret_crypto_btc_idr_indodax' if 'ret_crypto_btc_idr_indodax' in df_dl.columns else 'target_return_btc_idr'
assert target_ret_col in df_dl.columns, f'ASSERT STOP: Kolom target return tidak ditemukan di {dl_file}!'

# Merge TVP features using left join + forward-fill (exact v6 protocol)
df_merged = pd.merge(df_dl, df_tvp, on='Date', how='left').sort_values('Date')
tvp_cols = [c for c in df_tvp.columns if c != 'Date']
df_merged[tvp_cols] = df_merged[tvp_cols].ffill()
df = df_merged.dropna().reset_index(drop=True)

# Target: r_t^2 (log-return kuadrat BTC/IDR)
df['volatility_btc_idr'] = df[target_ret_col] ** 2
df['y'] = df['volatility_btc_idr'].values.astype('float32')

print(f'Total baris dataset setelah merge & ffill: {len(df)} baris.')
assert len(df) == 1676, f'ASSERT STOP: Total baris df harus 1676, ditemukan {len(df)}!'

# Verifikasi fitur eksogen
for c in SPILL_COLS:
    assert c in df.columns, f'ASSERT STOP: Fitur eksogen {c} tidak ditemukan di dataset!'

# Dataframe NeuralForecast
nf_df = pd.DataFrame()
nf_df['ds'] = pd.date_range(start='2020-01-01', periods=len(df), freq='D')
nf_df['unique_id'] = 'BTC_IDR'
nf_df['y'] = df['y'].values
for col in sorted(SPILL_COLS):
    nf_df[col] = df[col].values.astype('float32')
nf_df['real_date'] = df['Date'].values

# Split 1165 train, 167 val, 344 test
n_train = 1165
n_val = 167
n_test = 344
n_train_val = n_train + n_val

assert len(nf_df) >= (n_train_val + n_test), f'Panjang data {len(nf_df)} kurang untuk split 1165+167+344!'

df_train = nf_df.iloc[:n_train].copy()
df_val = nf_df.iloc[n_train:n_train_val].copy()
df_train_val = nf_df.iloc[:n_train_val].copy()
df_test = nf_df.iloc[n_train_val:n_train_val + n_test].copy()

# Leakage Assertions
assert len(df_train) == 1165, f'Train count {len(df_train)} != 1165'
assert len(df_val) == 167, f'Val count {len(df_val)} != 167'
assert len(df_train_val) == 1332, f'Train+Val count {len(df_train_val)} != 1332'
assert len(df_test) == 344, f'Test count {len(df_test)} != 344'
assert pd.to_datetime(df_train_val['real_date']).max() < pd.to_datetime('2025-04-24'), 'Leakage: train+val contains test dates!'

print(f'Train Data Range     : {pd.to_datetime(df_train["real_date"]).min().strftime("%Y-%m-%d")} s/d {pd.to_datetime(df_train["real_date"]).max().strftime("%Y-%m-%d")} (N={len(df_train)})')
print(f'Val Data Range       : {pd.to_datetime(df_val["real_date"]).min().strftime("%Y-%m-%d")} s/d {pd.to_datetime(df_val["real_date"]).max().strftime("%Y-%m-%d")} (N={len(df_val)})')
print(f'Test Target Range    : {pd.to_datetime(df_test["real_date"]).min().strftime("%Y-%m-%d")} s/d {pd.to_datetime(df_test["real_date"]).max().strftime("%Y-%m-%d")} (N={len(df_test)})')
print('✅ Data loading & Anti-Leakage Split Assertion PASS.')


=== HASIL PENEMUAN FILE DATASET (AUTO-DISCOVERY) ===
dataset_deep_learning.csv            : /kaggle/input/datasets/ferastoro/ai-stocks-crypto-and-ihsg/data_clean/dataset_deep_learning.csv
tvp_var_dynamic_features_causal.csv  : /kaggle/input/datasets/ferastoro/spillover-v5-inputs/rsic-v5-inputs/tvp_var_dynamic_features_causal.csv
Total baris dataset setelah merge & ffill: 1676 baris.
Train Data Range     : 2020-01-06 s/d 2024-08-21 (N=1165)
Val Data Range       : 2024-08-22 s/d 2025-04-23 (N=167)
Test Target Range    : 2025-04-24 s/d 2026-09-04 (N=344)
✅ Data loading & Anti-Leakage Split Assertion PASS.


# CELL 4: RESTORE BEST WEIGHTS CALLBACK & MODEL FACTORY (V6 TRAINER_KWARGS PATTERN)


In [4]:
from neuralforecast.losses.pytorch import MAE, MSE
from neuralforecast.models import TFT, NBEATSx

def get_weights_checksum_from_dict(state_dict) -> str:
    h = hashlib.md5()
    for k, v in sorted(state_dict.items()):
        h.update(k.encode('utf-8'))
        h.update(v.detach().cpu().numpy().tobytes())
    return h.hexdigest()

def get_weights_checksum(model) -> str:
    return get_weights_checksum_from_dict(model.state_dict())

class RestoreBestWeightsCallback(pl.Callback):
    """Custom Callback to track full validation check curve, train loss, and restore best weights."""
    def __init__(self, log_curve_file=None):
        super().__init__()
        self.best_score = None
        self.best_step = None
        self.best_state_dict = None
        self.best_checksum = None
        self.actual_final_step = 0
        self.stopped_epoch = 0
        self.es_triggered = False
        self.restored = False
        self.history = []  # list of dicts: {'check_step', 'val_loss', 'train_loss'}
        self.current_train_loss = np.nan
        self.train_loss_at_best_step = np.nan
        self.log_curve_file = log_curve_file

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        self.actual_final_step = trainer.global_step
        if isinstance(outputs, dict) and 'loss' in outputs:
            self.current_train_loss = float(outputs['loss'].detach().cpu().item())
        elif hasattr(trainer, 'callback_metrics') and 'train_loss' in trainer.callback_metrics:
            self.current_train_loss = float(trainer.callback_metrics['train_loss'])

    def on_validation_end(self, trainer, pl_module):
        if getattr(trainer, 'sanity_checking', False) or trainer.global_step == 0:
            return
        val_loss = trainer.callback_metrics.get('ptl/val_loss', None)
        if val_loss is not None:
            v_val = float(val_loss)
            self.history.append({
                'check_step': trainer.global_step,
                'val_loss': v_val,
                'train_loss': self.current_train_loss
            })
            if self.best_score is None or v_val < self.best_score:
                self.best_score = v_val
                self.best_step = trainer.global_step
                self.train_loss_at_best_step = self.current_train_loss
                self.best_state_dict = {k: v.detach().cpu().clone() for k, v in pl_module.state_dict().items()}
                self.best_checksum = get_weights_checksum_from_dict(self.best_state_dict)

    def on_train_end(self, trainer, pl_module):
        self.actual_final_step = trainer.global_step
        for cb in trainer.callbacks:
            if isinstance(cb, EarlyStopping):
                if cb.stopped_epoch > 0:
                    self.es_triggered = True
                    self.stopped_epoch = cb.stopped_epoch
        if self.best_state_dict is not None:
            pl_module.load_state_dict(self.best_state_dict)
            self.restored = True
        if self.log_curve_file and len(self.history) > 0:
            df_curve = pd.DataFrame(self.history)
            atomic_save_csv(df_curve, self.log_curve_file)

def get_model_instance(model_name: str,
                       fset_cols: list,
                       h: int = 10,
                       input_size: int = 50,
                       max_steps: int = 1000,
                       val_check_steps: int = 25,
                       patience: int = 5,
                       seed: int = 42,
                       loss_name: str = 'MSE',
                       extra_kwargs: dict = None,
                       log_curve_file: str = None):
    loss_inst = MAE() if loss_name == 'MAE' else MSE()
    tracker = RestoreBestWeightsCallback(log_curve_file=log_curve_file)
    
    base_kwargs = {
        'h': h,
        'input_size': input_size,
        'hist_exog_list': fset_cols,
        'max_steps': max_steps,
        'val_check_steps': val_check_steps,
        'early_stop_patience_steps': patience,
        'loss': loss_inst,
        'random_seed': seed,
        'scaler_type': 'standard'
    }
    if extra_kwargs:
        base_kwargs.update(extra_kwargs)

    if model_name == 'TFT':
        if 'hidden_size' not in base_kwargs:
            base_kwargs['hidden_size'] = 64
        model = TFT(**base_kwargs)
    elif model_name == 'NBEATSx':
        if 'mlp_units' not in base_kwargs:
            base_kwargs['mlp_units'] = [[256, 256], [256, 256], [256, 256]]
        model = NBEATSx(**base_kwargs)
    else:
        raise ValueError(f"Model {model_name} tidak didukung!")

    # Trainer configuration using update() - identical to v6
    model.trainer_kwargs.update({
        'enable_progress_bar': False,
        'enable_checkpointing': False,
        'enable_model_summary': False,
        'logger': False
    })
    if torch.cuda.is_available():
        model.trainer_kwargs['accelerator'] = 'cuda'
        model.trainer_kwargs['devices'] = 1
    else:
        model.trainer_kwargs['accelerator'] = 'cpu'

    if 'callbacks' not in model.trainer_kwargs or model.trainer_kwargs['callbacks'] is None:
        model.trainer_kwargs['callbacks'] = []
    model.trainer_kwargs['callbacks'].append(tracker)

    has_es = any(isinstance(cb, EarlyStopping) for cb in model.trainer_kwargs['callbacks'])
    assert has_es, "ASSERT STOP: EarlyStopping bawaan hilang dari trainer_kwargs['callbacks']!"

    return model


# CELL 5: EXPERIMENT A - UNTUNED LOSS-MATCHED RETRAINING (PRESERVED REPRODUCIBILITY)


In [5]:

from neuralforecast import NeuralForecast

exp_a_configs = [
    ('TFT', 'SPILL', 'MAE'),
    ('TFT', 'NOSPILL', 'MAE'),
    ('NBEATSx', 'SPILL', 'MSE'),
    ('NBEATSx', 'NOSPILL', 'MSE')
]

exp_a_seeds = SEEDS if STAGE != 'SMOKE' else [42]
a_measured_fit_times = []

if STAGE in ['A', 'SMOKE']:
    t0_stage_a = time.time()
    print('=' * 80)
    print(f'STARTING EXPERIMENT A: UNTUNED LOSS-MATCHED RETRAINING ({len(exp_a_configs)} configs x {len(exp_a_seeds)} seeds)')
    print('=' * 80)
    
    for m_name, fset_name, loss_choice in exp_a_configs:
        cfg_label = f'{m_name}_{loss_choice}_{fset_name}'
        fset_cols = FEATURE_SETS[fset_name]
        hp = V6_BEST_HP[m_name]
        extra_kw = {k: v for k, v in hp.items() if k not in ['input_size', 'max_steps', 'val_check_steps', 'patience']}
        
        for seed in exp_a_seeds:
            run_id = f"A_{cfg_label}_s{seed}"
            partial_file = os.path.join(OUTPUT_DIR, f'v7_expA_partial_{cfg_label}_s{seed}.csv')
            curve_file = os.path.join(LOGS_DIR, f'val_curve_{cfg_label}_s{seed}.csv')
            
            if os.path.exists(partial_file):
                print(f'[RESUME] Found existing checkpoint: {partial_file}. Skipping...')
                continue
                
            est_next_time = np.mean(a_measured_fit_times) if len(a_measured_fit_times) > 0 else 60.0
            if not check_runtime_budget(est_next_time):
                print(f"Stopping Experiment A to preserve budget. Remaining runs marked missing.")
                break
                
            set_all_seeds(seed)
            model = get_model_instance(
                model_name=m_name,
                fset_cols=fset_cols,
                h=H,
                input_size=hp['input_size'],
                max_steps=hp['max_steps'],
                val_check_steps=hp['val_check_steps'],
                patience=hp['patience'],
                seed=seed,
                loss_name=loss_choice,
                extra_kwargs=extra_kw,
                log_curve_file=curve_file
            )
            
            nf_sub = nf_df[['unique_id', 'ds', 'y'] + fset_cols].iloc[:n_train_val + N_TEST_CUTOFFS].copy()
            nf = NeuralForecast(models=[model], freq='D')
            
            t_fit_start = time.time()
            nf.fit(df=nf_sub.iloc[:n_train_val], val_size=n_val)
            t_fit = time.time() - t_fit_start
            a_measured_fit_times.append(t_fit)
            
            m_fitted = nf.models[0]
            chk_before = get_weights_checksum(m_fitted)
            
            tr_list = [cb for cb in m_fitted.trainer_kwargs.get('callbacks', []) if isinstance(cb, RestoreBestWeightsCallback)]
            tr = tr_list[0] if tr_list else None
            
            t_cv_start = time.time()
            cv_df = nf.cross_validation(
                df=nf_sub,
                n_windows=N_TEST_CUTOFFS,
                step_size=1,
                use_fitted=True
            ).reset_index(drop=True)
            t_cv = time.time() - t_cv_start
            
            chk_after = get_weights_checksum(m_fitted)
            assert chk_before == chk_after, f"ASSERT STOP: Model mutated during inference in {run_id}!"
            
            unique_cutoffs = sorted(cv_df['cutoff'].unique())
            test_target_subset = nf_df.iloc[n_train_val:n_train_val + N_TEST_CUTOFFS].reset_index(drop=True)
            cutoff_to_date = {c: test_target_subset.iloc[i]['real_date'] for i, c in enumerate(unique_cutoffs)}
            cv_df['real_date'] = cv_df['cutoff'].map(cutoff_to_date)
            cv_df['cutoff'] = cv_df['cutoff'].astype(str)
            cv_df['real_date'] = cv_df['real_date'].astype(str)
            
            atomic_save_csv(cv_df, partial_file)
            
            preds = cv_df[m_name].values
            neg_pct_all = float(np.mean(preds < 0) * 100.0)
            neg_pct_h1 = float(np.mean(cv_df.groupby('cutoff').head(1)[m_name].values < 0) * 100.0)
            neg_pct_h10 = float(np.mean(cv_df.groupby('cutoff').tail(1)[m_name].values < 0) * 100.0)
            
            log_run_metrics({
                'run_id': run_id,
                'stage': 'A',
                'config': cfg_label,
                'model': m_name,
                'feature_set': fset_name,
                'loss_name': loss_choice,
                'seed': seed,
                'fit_calls': 1,
                'best_step': tr.best_step if tr else np.nan,
                'stop_step': tr.actual_final_step if tr else np.nan,
                'train_loss_best_step': tr.train_loss_at_best_step if tr else np.nan,
                'val_loss_best_step': tr.best_score if tr else np.nan,
                'max_steps': hp['max_steps'],
                'val_check_steps': hp['val_check_steps'],
                'patience': hp['patience'],
                'weights_restored': tr.restored if tr else False,
                'checksum_before': chk_before,
                'checksum_after': chk_after,
                'checksum_match': (chk_before == chk_after),
                'fit_seconds': round(t_fit, 2),
                'cv_seconds': round(t_cv, 2),
                'neg_pct_all': neg_pct_all,
                'neg_pct_h1': neg_pct_h1,
                'neg_pct_h10': neg_pct_h10,
                'partial_file': os.path.basename(partial_file)
            })
            print(f"[FINISHED A] {run_id:<28} BestStep={tr.best_step if tr else 0:<4} StopStep={tr.actual_final_step if tr else 0:<4} Neg%={neg_pct_all:.2f}% Time={t_fit:.1f}s+{t_cv:.1f}s")
    STAGE_TIMINGS['A'] = time.time() - t0_stage_a


# CELL 6: EXPERIMENT A_TUNED - PROPERLY TUNED LOSS-MATCHED RETRAINING


In [6]:
from sklearn.metrics import mean_squared_error

a_tuned_models = ['NBEATSx']
if RUN_TFT_MAE_TUNED:
    a_tuned_models.append('TFT')

# v7 A_TUNED Grid (8 combos per model): input_size in {20, 50} x lr in {1e-3, 7.5e-4, 5e-4, 3e-4}
# Loss fixed to MSE for NBEATSx, MAE for TFT.
# Difference from v6 Stage B: v6 varied loss {MAE, MSE} across 2 lr levels {1e-3, 3e-4};
# v7 fixes loss to match target evaluation loss and densifies lr exploration (4 rates vs 2 rates in v6).
# v7 is a loss-matched densified grid, NOT the same grid as v6.
# v7 A_TUNED Grids:
# - NBEATSx (12 combos): input_size {20, 50, 100} x lr {1e-3, 5e-4, 3e-4, 1e-4}, MSE loss only.
# - TFT (8 combos)    : input_size {20, 50} x lr {1e-3, 7.5e-4, 5e-4, 3e-4}, MAE loss only.
# Both models ranked on 2-seed validation MSE average (seeds 42 and 7, over h=1 and h=10).
TUNING_GRIDS = {
    'NBEATSx': [
        {'input_size': 20, 'learning_rate': 1e-3, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MSE'},
        {'input_size': 20, 'learning_rate': 5e-4, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MSE'},
        {'input_size': 20, 'learning_rate': 3e-4, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MSE'},
        {'input_size': 20, 'learning_rate': 1e-4, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MSE'},
        {'input_size': 50, 'learning_rate': 1e-3, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MSE'},
        {'input_size': 50, 'learning_rate': 5e-4, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MSE'},
        {'input_size': 50, 'learning_rate': 3e-4, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MSE'},
        {'input_size': 50, 'learning_rate': 1e-4, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MSE'},
        {'input_size': 100, 'learning_rate': 1e-3, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MSE'},
        {'input_size': 100, 'learning_rate': 5e-4, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MSE'},
        {'input_size': 100, 'learning_rate': 3e-4, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MSE'},
        {'input_size': 100, 'learning_rate': 1e-4, 'mlp_units': [[256, 256], [256, 256], [256, 256]], 'loss_name': 'MSE'},
    ],
    'TFT': [
        {'input_size': 20, 'learning_rate': 1e-3, 'hidden_size': 64, 'loss_name': 'MAE'},
        {'input_size': 20, 'learning_rate': 7.5e-4, 'hidden_size': 64, 'loss_name': 'MAE'},
        {'input_size': 20, 'learning_rate': 5e-4, 'hidden_size': 64, 'loss_name': 'MAE'},
        {'input_size': 20, 'learning_rate': 3e-4, 'hidden_size': 64, 'loss_name': 'MAE'},
        {'input_size': 50, 'learning_rate': 1e-3, 'hidden_size': 64, 'loss_name': 'MAE'},
        {'input_size': 50, 'learning_rate': 7.5e-4, 'hidden_size': 64, 'loss_name': 'MAE'},
        {'input_size': 50, 'learning_rate': 5e-4, 'hidden_size': 64, 'loss_name': 'MAE'},
        {'input_size': 50, 'learning_rate': 3e-4, 'hidden_size': 64, 'loss_name': 'MAE'},
    ]
}

grid_eval_file = os.path.join(OUTPUT_DIR, 'a_tuned_grid_evaluation.csv')
best_tuned_hp_file = os.path.join(OUTPUT_DIR, 'v7_best_tuned_hyperparameters.json')
a_tuned_measured_fit_times = []

if STAGE in ['A_TUNED', 'ALL', 'SMOKE']:
    t0_stage_atuned = time.time()
    print('=' * 80)
    print(f'STARTING EXPERIMENT A_TUNED: HYPERPARAMETER TUNING & 5-SEED EVALUATION')
    print('=' * 80)
    
    # Anti-leakage assertion: tuning ONLY on train+val (1332 rows, date < 2025-04-24)
    assert len(df_train_val) == 1332, f"ASSERT STOP: df_train_val has {len(df_train_val)} rows, expected 1332!"
    assert pd.to_datetime(df_train_val['real_date']).max() < pd.to_datetime('2025-04-24'),         f"ASSERT STOP: Leakage! df_train_val contains test dates >= 2025-04-24 (max date: {df_train_val['real_date'].max()})"
    print("✅ Anti-Leakage Assertion PASS: Tuning evaluates strictly on 1,332 train+val rows (zero test set access).")
    
    grid_records = []
    best_tuned_hp = {}
    winner_edge_notes = {}
    
    stage_b_tuning_seeds = [42, 7] if STAGE != 'SMOKE' else [42]
    
    for m_name in a_tuned_models:
        grid = TUNING_GRIDS[m_name]
        if STAGE == 'SMOKE':
            if m_name == 'NBEATSx':
                # Exactly 2 combos: first combo (input=20, lr=1e-3) and extreme corner combo (input=100, lr=1e-4)
                corner_combo = [c for c in grid if c['input_size'] == 100 and c['learning_rate'] == 1e-4 and c['loss_name'] == 'MSE']
                grid = [grid[0]] + (corner_combo if corner_combo else [grid[-1]])  # exactly 2 combos
            else:
                grid = [grid[0]]  # TFT stays at 1 combo
            
        fset_cols = FEATURE_SETS['NOSPILL']
        
        # Hyperparameter check steps and patience configuration:
        # For NBEATSx: val_check_steps = 5 (v6 used 25).
        # Scaled patience = 25 checks (25 * 5 = 125 steps), keeping the exact same 125-step patience as v6 (5 * 25 = 125).
        # For TFT: keeps default VAL_CHECK_STEPS_BANNER (25) and PATIENCE_BANNER (5).
        if m_name == 'NBEATSx':
            tune_val_check_steps = 5
            tune_patience = 25 if STAGE != 'SMOKE' else 4
        else:
            tune_val_check_steps = VAL_CHECK_STEPS_BANNER
            tune_patience = PATIENCE_BANNER
            
        print(f"\n--- Tuning Grid for {m_name} ({len(grid)} combos, Seeds {stage_b_tuning_seeds}, val_check_steps={tune_val_check_steps}, patience={tune_patience}) ---")
        
        full_grid_spec = TUNING_GRIDS[m_name]
        min_grid_input = min(c['input_size'] for c in full_grid_spec)
        max_grid_input = max(c['input_size'] for c in full_grid_spec)
        min_grid_lr = min(c['learning_rate'] for c in full_grid_spec)
        max_grid_lr = max(c['learning_rate'] for c in full_grid_spec)
        
        for idx, hp_cand in enumerate(grid):
            combo_id = f"{m_name}_tuned_cfg{idx+1}"
            loss_choice = hp_cand['loss_name']
            extra_kw = {k: v for k, v in hp_cand.items() if k not in ['input_size', 'loss_name']}
            
            seed_val_mses = []
            seed_val_ratios = []
            seed_best_steps = []
            seed_stop_steps = []
            combo_fit_secs = 0.0
            
            for t_seed in stage_b_tuning_seeds:
                curve_file = os.path.join(LOGS_DIR, f'val_curve_tuning_{combo_id}_s{t_seed}.csv')
                set_all_seeds(t_seed)
                model = get_model_instance(
                    model_name=m_name,
                    fset_cols=fset_cols,
                    h=H,
                    input_size=hp_cand['input_size'],
                    max_steps=MAX_STEPS_BANNER,
                    val_check_steps=tune_val_check_steps,
                    patience=tune_patience,
                    seed=t_seed,
                    loss_name=loss_choice,
                    extra_kwargs=extra_kw,
                    log_curve_file=curve_file
                )
                
                nf_sub = df_train_val[['unique_id', 'ds', 'y'] + fset_cols].copy()
                nf = NeuralForecast(models=[model], freq='D')
                
                t0 = time.time()
                nf.fit(df=nf_sub, val_size=n_val)
                t_fit = time.time() - t0
                combo_fit_secs += t_fit
                a_tuned_measured_fit_times.append(t_fit)
                
                m_fitted = nf.models[0]
                chk_before = get_weights_checksum(m_fitted)
                tr_list = [cb for cb in m_fitted.trainer_kwargs.get('callbacks', []) if isinstance(cb, RestoreBestWeightsCallback)]
                tr = tr_list[0] if tr_list else None
                
                n_val_windows = n_val - H + 1
                cv_val = nf.cross_validation(
                    df=nf_sub,
                    n_windows=n_val_windows,
                    step_size=1,
                    use_fitted=True
                ).reset_index(drop=True)
                
                chk_after = get_weights_checksum(m_fitted)
                assert chk_before == chk_after, f"ASSERT STOP: Weights changed during cv_val in {combo_id}_s{t_seed}!"
                
                grp_v1 = cv_val.groupby('cutoff').head(1)
                grp_v10 = cv_val.groupby('cutoff').tail(1)
                mse_v1 = float(mean_squared_error(grp_v1['y'], grp_v1[m_name]))
                mse_v10 = float(mean_squared_error(grp_v10['y'], grp_v10[m_name]))
                mse_val_seed_avg = (mse_v1 + mse_v10) / 2.0
                
                val_mean_target = float(cv_val['y'].mean())
                val_mean_pred = float(cv_val[m_name].mean())
                val_ratio = val_mean_pred / val_mean_target if val_mean_target > 0 else np.nan
                
                seed_val_mses.append(mse_val_seed_avg)
                seed_val_ratios.append(val_ratio)
                seed_best_steps.append(tr.best_step if tr else np.nan)
                seed_stop_steps.append(tr.actual_final_step if tr else np.nan)
            
            # 2-Seed Mean Validation MSE (averaging over seeds 42 and 7 as in v6 Stage B)
            val_mse_avg_2seeds = float(np.mean(seed_val_mses))
            mean_val_ratio = float(np.mean(seed_val_ratios))
            
            ratio_flag = "OK"
            if mean_val_ratio > 2.0:
                ratio_flag = "FLAG_OVERSHOOT (>2x)"
            elif mean_val_ratio < 0.5:
                ratio_flag = "FLAG_UNDERSHOOT (<0.5x)"
                
            rec = {
                'model': m_name,
                'combo_id': combo_id,
                'input_size': hp_cand['input_size'],
                'learning_rate': hp_cand['learning_rate'],
                'loss_name': loss_choice,
                'val_check_steps': tune_val_check_steps,
                'patience': tune_patience,
                'hyperparameters': json.dumps(hp_cand),
                'best_step_s42': seed_best_steps[0] if len(seed_best_steps) > 0 else np.nan,
                'best_step_s7': seed_best_steps[1] if len(seed_best_steps) > 1 else np.nan,
                'val_mse_s42': seed_val_mses[0] if len(seed_val_mses) > 0 else np.nan,
                'val_mse_s7': seed_val_mses[1] if len(seed_val_mses) > 1 else np.nan,
                'val_mse_avg': val_mse_avg_2seeds,
                'val_forecast_to_target_ratio': mean_val_ratio,
                'ratio_flag': ratio_flag,
                'fit_seconds_total': round(combo_fit_secs, 2)
            }
            grid_records.append(rec)
            print(f"  [{combo_id}] Inp={hp_cand['input_size']} LR={hp_cand['learning_rate']:.1e} | 2-Seed ValMSE={val_mse_avg_2seeds:.6e} (s42={rec['val_mse_s42']:.6e}, s7={rec['val_mse_s7']:.6e}) | Ratio={mean_val_ratio:.2f} ({ratio_flag})")
            
        df_grid = pd.DataFrame(grid_records)
        grid_header_comment = """# RSIC v7 A_TUNED Hyperparameter Tuning Grid Evaluation
# v7 NBEATSx Grid : input_size in {20, 50, 100} x learning_rate in {1e-3, 5e-4, 3e-4, 1e-4} | Loss fixed to MSE (12 combos)
# v7 TFT Grid     : input_size in {20, 50} x learning_rate in {1e-3, 7.5e-4, 5e-4, 3e-4} | Loss fixed to MAE (8 combos)
# Selection Metric: Ranked by mean validation MSE across seeds 42 and 7 (averaged over h=1 and h=10).
# v6 Grid Diff    : v6 used {20, 50} x {3e-4, 1e-3} x {MAE, MSE}; v7 fixes loss and explores larger/finer grid with val_check_steps=5, patience=25 for NBEATSx.
"""
        tmp_grid_path = f"{grid_eval_file}.tmp_{os.getpid()}_{random.randint(1000, 9999)}"
        with open(tmp_grid_path, 'w') as f_g:
            f_g.write(grid_header_comment)
            df_grid.to_csv(f_g, index=False)
        os.replace(tmp_grid_path, grid_eval_file)
        
        df_m = df_grid[df_grid['model'] == m_name].sort_values('val_mse_avg').reset_index(drop=True)
        winner = df_m.iloc[0]
        winner_hp = json.loads(winner['hyperparameters'])
        # Store tuned trainer kwargs into winner hyperparameters
        winner_hp['val_check_steps'] = tune_val_check_steps
        winner_hp['patience'] = tune_patience
        best_tuned_hp[m_name] = winner_hp
        
        # Edge checks (specifically for input 100 or lr 1e-4)
        is_edge_input_100 = (winner_hp['input_size'] == 100)
        is_edge_lr_1e4 = (winner_hp['learning_rate'] == 1e-4)
        is_edge_input = (winner_hp['input_size'] in [min_grid_input, max_grid_input])
        is_edge_lr = (winner_hp['learning_rate'] in [min_grid_lr, max_grid_lr])
        is_edge = is_edge_input or is_edge_lr
        
        edge_notes = []
        if is_edge_input_100:
            edge_notes.append("input_size=100 (upper bound)")
        elif winner_hp['input_size'] == min_grid_input:
            edge_notes.append(f"input_size={min_grid_input} (lower bound)")
        if is_edge_lr_1e4:
            edge_notes.append("learning_rate=1e-4 (lower bound)")
        elif winner_hp['learning_rate'] == max_grid_lr:
            edge_notes.append(f"learning_rate={max_grid_lr} (upper bound)")
            
        edge_status_str = f"ON THE EDGE ({', '.join(edge_notes)})" if is_edge else "INTERIOR (neither input 100 nor lr 1e-4)"
        winner_edge_notes[m_name] = edge_status_str
        
        print(f"\n🏆 Winner for {m_name}: {winner['combo_id']}")
        print(f"   Hyperparameters: {winner_hp}")
        print(f"   2-Seed Val MSE Avg: {winner['val_mse_avg']:.6e} | Ratio: {winner['val_forecast_to_target_ratio']:.2f}")
        print(f"   Grid Edge Check   : {edge_status_str}")
        
    with open(best_tuned_hp_file, 'w') as f_hp:
        json.dump(best_tuned_hp, f_hp, indent=2)
    print(f"✅ Saved tuned hyperparameters to: {best_tuned_hp_file}")
    
    # ----------------------------------------------------------------------------------
    # RUN WINNER ACROSS 5 SEEDS x SPILL/NOSPILL ON TEST CUTOFFS
    # ----------------------------------------------------------------------------------
    tuned_seeds = SEEDS if STAGE != 'SMOKE' else [42]
    tuned_configs = []
    for m_name in a_tuned_models:
        for fset in ['SPILL', 'NOSPILL']:
            cfg_lbl = f"{m_name}_MSEtuned_{fset}" if m_name == 'NBEATSx' else f"{m_name}_MAEtuned_{fset}"
            tuned_configs.append((m_name, fset, cfg_lbl, best_tuned_hp[m_name]))
            
    print(f"\n--- Running Tuned Winners Across {len(tuned_seeds)} Seeds x {len(tuned_configs)} Configs ---")
    
    for m_name, fset_name, cfg_label, hp_winner in tuned_configs:
        fset_cols = FEATURE_SETS[fset_name]
        loss_choice = hp_winner['loss_name']
        extra_kw = {k: v for k, v in hp_winner.items() if k not in ['input_size', 'loss_name', 'max_steps', 'val_check_steps', 'patience']}
        
        for seed in tuned_seeds:
            run_id = f"A_TUNED_{cfg_label}_s{seed}"
            partial_file = os.path.join(OUTPUT_DIR, f'v7_expAtuned_partial_{cfg_label}_s{seed}.csv')
            curve_file = os.path.join(LOGS_DIR, f'val_curve_{cfg_label}_s{seed}.csv')
            
            if os.path.exists(partial_file):
                print(f"[RESUME] Found existing partial: {partial_file}. Skipping...")
                continue
                
            est_time = np.mean(a_tuned_measured_fit_times) if len(a_tuned_measured_fit_times) > 0 else 60.0
            if not check_runtime_budget(est_time):
                print(f"Stopping A_TUNED to preserve budget. Remaining runs marked missing.")
                break
                
            set_all_seeds(seed)
            # Winner final run uses the same val_check_steps and patience as tuning
            w_val_check_steps = hp_winner.get('val_check_steps', VAL_CHECK_STEPS_BANNER)
            w_patience = hp_winner.get('patience', PATIENCE_BANNER)
            model = get_model_instance(
                model_name=m_name,
                fset_cols=fset_cols,
                h=H,
                input_size=hp_winner['input_size'],
                max_steps=MAX_STEPS_BANNER,
                val_check_steps=w_val_check_steps,
                patience=w_patience,
                seed=seed,
                loss_name=loss_choice,
                extra_kwargs=extra_kw,
                log_curve_file=curve_file
            )
            
            nf_sub = nf_df[['unique_id', 'ds', 'y'] + fset_cols].iloc[:n_train_val + N_TEST_CUTOFFS].copy()
            nf = NeuralForecast(models=[model], freq='D')
            
            t0 = time.time()
            nf.fit(df=nf_sub.iloc[:n_train_val], val_size=n_val)
            t_fit = time.time() - t0
            a_tuned_measured_fit_times.append(t_fit)
            
            m_fitted = nf.models[0]
            chk_before = get_weights_checksum(m_fitted)
            tr_list = [cb for cb in m_fitted.trainer_kwargs.get('callbacks', []) if isinstance(cb, RestoreBestWeightsCallback)]
            tr = tr_list[0] if tr_list else None
            
            t_cv_start = time.time()
            cv_df = nf.cross_validation(
                df=nf_sub,
                n_windows=N_TEST_CUTOFFS,
                step_size=1,
                use_fitted=True
            ).reset_index(drop=True)
            t_cv = time.time() - t_cv_start
            
            chk_after = get_weights_checksum(m_fitted)
            assert chk_before == chk_after, f"ASSERT STOP: Weights changed during CV in {run_id}!"
            
            unique_cutoffs = sorted(cv_df['cutoff'].unique())
            test_target_subset = nf_df.iloc[n_train_val:n_train_val + N_TEST_CUTOFFS].reset_index(drop=True)
            cutoff_to_date = {c: test_target_subset.iloc[i]['real_date'] for i, c in enumerate(unique_cutoffs)}
            cv_df['real_date'] = cv_df['cutoff'].map(cutoff_to_date)
            cv_df['cutoff'] = cv_df['cutoff'].astype(str)
            cv_df['real_date'] = cv_df['real_date'].astype(str)
            
            atomic_save_csv(cv_df, partial_file)
            
            preds = cv_df[m_name].values
            neg_pct_all = float(np.mean(preds < 0) * 100.0)
            neg_pct_h1 = float(np.mean(cv_df.groupby('cutoff').head(1)[m_name].values < 0) * 100.0)
            neg_pct_h10 = float(np.mean(cv_df.groupby('cutoff').tail(1)[m_name].values < 0) * 100.0)
            
            log_run_metrics({
                'run_id': run_id,
                'stage': 'A_TUNED',
                'config': cfg_label,
                'model': m_name,
                'feature_set': fset_name,
                'loss_name': loss_choice,
                'seed': seed,
                'fit_calls': 1,
                'best_step': tr.best_step if tr else np.nan,
                'stop_step': tr.actual_final_step if tr else np.nan,
                'train_loss_best_step': tr.train_loss_at_best_step if tr else np.nan,
                'val_loss_best_step': tr.best_score if tr else np.nan,
                'max_steps': MAX_STEPS_BANNER,
                'val_check_steps': w_val_check_steps,
                'patience': w_patience,
                'weights_restored': tr.restored if tr else False,
                'checksum_before': chk_before,
                'checksum_after': chk_after,
                'checksum_match': (chk_before == chk_after),
                'fit_seconds': round(t_fit, 2),
                'cv_seconds': round(t_cv, 2),
                'neg_pct_all': neg_pct_all,
                'neg_pct_h1': neg_pct_h1,
                'neg_pct_h10': neg_pct_h10,
                'partial_file': os.path.basename(partial_file)
            })
            print(f"[FINISHED A_TUNED] {run_id:<32} BestStep={tr.best_step if tr else 0:<4} StopStep={tr.actual_final_step if tr else 0:<4} Neg%={neg_pct_all:.2f}% Time={t_fit:.1f}s+{t_cv:.1f}s")
    STAGE_TIMINGS['A_TUNED'] = time.time() - t0_stage_atuned


Seed set to 42
Seed set to 42


STARTING EXPERIMENT A_TUNED: HYPERPARAMETER TUNING & 5-SEED EVALUATION
✅ Anti-Leakage Assertion PASS: Tuning evaluates strictly on 1,332 train+val rows (zero test set access).

--- Tuning Grid for NBEATSx (12 combos, Seeds [42, 7], val_check_steps=5, patience=25) ---


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Seed set to 7
Seed set to 7
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Seed set to 42
Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


  [NBEATSx_tuned_cfg1] Inp=20 LR=1.0e-03 | 2-Seed ValMSE=5.397029e-06 (s42=5.358813e-06, s7=5.435245e-06) | Ratio=2.70 (FLAG_OVERSHOOT (>2x))


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Seed set to 7
Seed set to 7
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Seed set to 42
Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


  [NBEATSx_tuned_cfg2] Inp=20 LR=5.0e-04 | 2-Seed ValMSE=3.993379e-06 (s42=3.954442e-06, s7=4.032317e-06) | Ratio=2.07 (FLAG_OVERSHOOT (>2x))


... (72 outputs left out)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Seed set to 99
Seed set to 99
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


[FINISHED A_TUNED] A_TUNED_TFT_MAEtuned_NOSPILL_s21 BestStep=50   StopStep=175  Neg%=2.42% Time=38.0s+0.2s


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


[FINISHED A_TUNED] A_TUNED_TFT_MAEtuned_NOSPILL_s99 BestStep=100  StopStep=225  Neg%=0.00% Time=48.8s+0.2s


# CELL 7: EXPERIMENT B - 60-DAY EXPANDING-WINDOW PERIODIC REFITTING (SEED 42)


In [7]:

REFIT_STEP = 60
exp_b_configs = [
    ('TFT', 'SPILL', 'MSE'),
    ('TFT', 'NOSPILL', 'MSE'),
    ('NBEATSx', 'SPILL', 'MAE'),
    ('NBEATSx', 'NOSPILL', 'MAE')
]

total_test_cutoffs = N_TEST_CUTOFFS
blocks = []
for b_idx, start_c in enumerate(range(0, total_test_cutoffs, REFIT_STEP)):
    end_c = min(start_c + REFIT_STEP, total_test_cutoffs)
    blocks.append((b_idx + 1, start_c, end_c))

b_measured_fit_times = []

if STAGE in ['B', 'ALL', 'SMOKE']:
    t0_stage_b = time.time()
    print('=' * 80)
    print(f'STARTING EXPERIMENT B: 60-DAY EXPANDING-WINDOW REFIT ({len(blocks)} blocks, Seed 42)')
    print('=' * 80)
    print('Refit Origins & Training Expansions:')
    for b_num, start_c, end_c in blocks:
        origin_idx = n_train_val + start_c
        origin_date = nf_df.iloc[origin_idx - 1]['real_date'].strftime('%Y-%m-%d')
        first_pred_date = nf_df.iloc[origin_idx]['real_date'].strftime('%Y-%m-%d')
        n_cutoffs_block = end_c - start_c
        print(f"  Block {b_num}: Cutoffs {start_c:3d}..{end_c-1:3d} ({n_cutoffs_block:2d} cutoffs) | Train Rows: {origin_idx:4d} | Origin: {origin_date} | First Pred: {first_pred_date}")
        
    for m_name, fset_name, loss_choice in exp_b_configs:
        cfg_label = f'{m_name}_{loss_choice}_{fset_name}'
        fset_cols = FEATURE_SETS[fset_name]
        hp = V6_BEST_HP[m_name]
        extra_kw = {k: v for k, v in hp.items() if k not in ['input_size', 'max_steps', 'val_check_steps', 'patience']}
        
        for b_num, start_c, end_c in blocks:
            n_cutoffs_block = end_c - start_c
            run_id = f"B_{cfg_label}_blk{b_num}"
            partial_file = os.path.join(OUTPUT_DIR, f'v7_expB_partial_{cfg_label}_blk{b_num}.csv')
            curve_file = os.path.join(LOGS_DIR, f'val_curve_{cfg_label}_blk{b_num}.csv')
            
            if os.path.exists(partial_file):
                print(f"[RESUME] Found existing checkpoint: {partial_file}. Skipping...")
                continue
                
            est_b_time = np.mean(b_measured_fit_times) if len(b_measured_fit_times) > 0 else 90.0
            if not check_runtime_budget(est_b_time):
                print(f"Stopping Experiment B to preserve budget. Remaining blocks marked missing.")
                break
                
            origin_idx = n_train_val + start_c
            df_train_b = nf_df.iloc[:origin_idx].copy()
            df_eval_b = nf_df.iloc[:origin_idx + n_cutoffs_block].copy()
            
            assert len(df_train_b) == origin_idx, f"Train length {len(df_train_b)} != {origin_idx}"
            assert df_train_b['ds'].max() <= nf_df.iloc[origin_idx - 1]['ds'], "LEAKAGE: max(train ds) > origin ds!"
            assert df_eval_b.iloc[origin_idx]['real_date'] > df_train_b.iloc[-1]['real_date'], "LEAKAGE: target date <= origin date!"
            
            set_all_seeds(42)
            model = get_model_instance(
                model_name=m_name,
                fset_cols=fset_cols,
                h=H,
                input_size=hp['input_size'],
                max_steps=hp['max_steps'],
                val_check_steps=hp['val_check_steps'],
                patience=hp['patience'],
                seed=42,
                loss_name=loss_choice,
                extra_kwargs=extra_kw,
                log_curve_file=curve_file
            )
            
            nf_b = NeuralForecast(models=[model], freq='D')
            
            t0 = time.time()
            nf_b.fit(df=df_train_b[['unique_id', 'ds', 'y'] + fset_cols], val_size=n_val)
            t_fit = time.time() - t0
            b_measured_fit_times.append(t_fit)
            
            m_fitted = nf_b.models[0]
            chk_before = get_weights_checksum(m_fitted)
            tr_list = [cb for cb in m_fitted.trainer_kwargs.get('callbacks', []) if isinstance(cb, RestoreBestWeightsCallback)]
            tr = tr_list[0] if tr_list else None
            
            t_cv_start = time.time()
            cv_block = nf_b.cross_validation(
                df=df_eval_b[['unique_id', 'ds', 'y'] + fset_cols],
                n_windows=n_cutoffs_block,
                step_size=1,
                use_fitted=True
            ).reset_index(drop=True)
            t_cv = time.time() - t_cv_start
            
            chk_after = get_weights_checksum(m_fitted)
            assert chk_before == chk_after, f"ASSERT STOP: Weights mutated in {run_id}!"
            
            unique_cutoffs = sorted(cv_block['cutoff'].unique())
            test_target_block = nf_df.iloc[origin_idx:origin_idx + n_cutoffs_block].reset_index(drop=True)
            cutoff_to_date = {c: test_target_block.iloc[i]['real_date'] for i, c in enumerate(unique_cutoffs)}
            cv_block['real_date'] = cv_block['cutoff'].map(cutoff_to_date)
            cv_block['cutoff'] = cv_block['cutoff'].astype(str)
            cv_block['real_date'] = cv_block['real_date'].astype(str)
            cv_block['block_id'] = b_num
            
            atomic_save_csv(cv_block, partial_file)
            
            preds = cv_block[m_name].values
            neg_pct_all = float(np.mean(preds < 0) * 100.0)
            neg_pct_h1 = float(np.mean(cv_block.groupby('cutoff').head(1)[m_name].values < 0) * 100.0)
            neg_pct_h10 = float(np.mean(cv_block.groupby('cutoff').tail(1)[m_name].values < 0) * 100.0)
            
            log_run_metrics({
                'run_id': run_id,
                'stage': 'B',
                'config': cfg_label,
                'model': m_name,
                'feature_set': fset_name,
                'loss_name': loss_choice,
                'seed': 42,
                'fit_calls': 1,
                'best_step': tr.best_step if tr else np.nan,
                'stop_step': tr.actual_final_step if tr else np.nan,
                'train_loss_best_step': tr.train_loss_at_best_step if tr else np.nan,
                'val_loss_best_step': tr.best_score if tr else np.nan,
                'max_steps': hp['max_steps'],
                'val_check_steps': hp['val_check_steps'],
                'patience': hp['patience'],
                'weights_restored': tr.restored if tr else False,
                'checksum_before': chk_before,
                'checksum_after': chk_after,
                'checksum_match': (chk_before == chk_after),
                'fit_seconds': round(t_fit, 2),
                'cv_seconds': round(t_cv, 2),
                'neg_pct_all': neg_pct_all,
                'neg_pct_h1': neg_pct_h1,
                'neg_pct_h10': neg_pct_h10,
                'partial_file': os.path.basename(partial_file)
            })
            print(f"[FINISHED B] {run_id:<30} BestStep={tr.best_step if tr else 0:<4} StopStep={tr.actual_final_step if tr else 0:<4} Neg%={neg_pct_all:.2f}% Time={t_fit:.1f}s+{t_cv:.1f}s")
    STAGE_TIMINGS['B'] = time.time() - t0_stage_b


Seed set to 42
Seed set to 42


STARTING EXPERIMENT B: 60-DAY EXPANDING-WINDOW REFIT (6 blocks, Seed 42)
Refit Origins & Training Expansions:
  Block 1: Cutoffs   0.. 59 (60 cutoffs) | Train Rows: 1332 | Origin: 2025-04-23 | First Pred: 2025-04-24
  Block 2: Cutoffs  60..119 (60 cutoffs) | Train Rows: 1392 | Origin: 2025-07-21 | First Pred: 2025-07-22
  Block 3: Cutoffs 120..179 (60 cutoffs) | Train Rows: 1452 | Origin: 2025-10-14 | First Pred: 2025-10-15
  Block 4: Cutoffs 180..239 (60 cutoffs) | Train Rows: 1512 | Origin: 2026-01-09 | First Pred: 2026-01-12
  Block 5: Cutoffs 240..299 (60 cutoffs) | Train Rows: 1572 | Origin: 2026-04-08 | First Pred: 2026-04-09
  Block 6: Cutoffs 300..334 (35 cutoffs) | Train Rows: 1632 | Origin: 2026-07-06 | First Pred: 2026-07-07


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Seed set to 42
Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


[FINISHED B] B_TFT_MSE_SPILL_blk1           BestStep=50   StopStep=175  Neg%=0.00% Time=70.1s+0.1s


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Seed set to 42
Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


[FINISHED B] B_TFT_MSE_SPILL_blk2           BestStep=25   StopStep=150  Neg%=0.00% Time=60.1s+0.1s


... (40 outputs left out)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Seed set to 42
Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


[FINISHED B] B_NBEATSx_MAE_NOSPILL_blk5     BestStep=50   StopStep=175  Neg%=8.17% Time=1.9s+0.1s


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


[FINISHED B] B_NBEATSx_MAE_NOSPILL_blk6     BestStep=50   StopStep=175  Neg%=1.71% Time=1.9s+0.1s


# CELL 8: NEGATIVE TEST FOR EXPERIMENT B ANTI-LEAKAGE ASSERTION (CPU SANITY CHECK)


In [8]:
print("=== NEGATIVE TEST: VERIFYING ANTI-LEAKAGE ASSERTION ON DELIBERATE CONTAMINATION ===")

leakage_caught = False
try:
    simulated_origin = 1332
    simulated_contaminated_train_max = 1333
    assert simulated_contaminated_train_max <= simulated_origin,         f"LEAKAGE DETECTED: max train index {simulated_contaminated_train_max} > origin {simulated_origin}"
except AssertionError as e:
    leakage_caught = True
    print(f"✅ Negative test PASS: assertion successfully caught deliberate leakage:\n   {e}")

assert leakage_caught, "ASSERT STOP: Negative test failed to catch simulated leakage!"


=== NEGATIVE TEST: VERIFYING ANTI-LEAKAGE ASSERTION ON DELIBERATE CONTAMINATION ===
✅ Negative test PASS: assertion successfully caught deliberate leakage:
   LEAKAGE DETECTED: max train index 1333 > origin 1332


# CELL 9: AGREGASI PREDIKSI ENSEMBLE KE FORMAT V6_ENSEMBLE_PREDICTIONS.CSV


In [9]:
print('=== AGREGASI PREDIKSI ENSEMBLE KE FORMAT V6 ===')

ensemble_rows = []

# 1. Experiment A Ensembles (Averaging over 5 seeds)
for m_name, fset_name, loss_choice in exp_a_configs:
    cfg_label = f'{m_name}_{loss_choice}_{fset_name}'
    seed_dfs = []
    for seed in exp_a_seeds:
        p_file = os.path.join(OUTPUT_DIR, f'v7_expA_partial_{cfg_label}_s{seed}.csv')
        if os.path.exists(p_file):
            seed_dfs.append(pd.read_csv(p_file))
            
    if len(seed_dfs) > 0:
        base_df = seed_dfs[0][['cutoff', 'real_date', 'unique_id', 'ds', 'y']].copy()
        pred_matrix = np.column_stack([sdf[m_name].values for sdf in seed_dfs])
        base_df['prediction'] = pred_matrix.mean(axis=1)
        if len(seed_dfs) == 5:
            base_df['Config'] = cfg_label
            print(f"  Ensemble Exp A: {cfg_label:<24} averaged across 5 seeds ({len(base_df)} rows)")
        else:
            partial_cfg_name = f"{cfg_label}_partial{len(seed_dfs)}seeds"
            base_df['Config'] = partial_cfg_name
            PARTIAL_ENSEMBLES.append(f"{partial_cfg_name} ({len(seed_dfs)}/5 seeds completed)")
            print(f"  ⚠️ Partial Ensemble Exp A: {partial_cfg_name:<30} averaged across {len(seed_dfs)}/5 seeds ({len(base_df)} rows)")
        base_df['Stage'] = 'A'
        ensemble_rows.append(base_df)

# 2. Experiment A_TUNED Ensembles (Averaging over 5 seeds)
if 'tuned_configs' in locals():
    for m_name, fset_name, cfg_label, _ in tuned_configs:
        seed_dfs = []
        for seed in tuned_seeds:
            p_file = os.path.join(OUTPUT_DIR, f'v7_expAtuned_partial_{cfg_label}_s{seed}.csv')
            if os.path.exists(p_file):
                seed_dfs.append(pd.read_csv(p_file))
                
        if len(seed_dfs) > 0:
            base_df = seed_dfs[0][['cutoff', 'real_date', 'unique_id', 'ds', 'y']].copy()
            pred_matrix = np.column_stack([sdf[m_name].values for sdf in seed_dfs])
            base_df['prediction'] = pred_matrix.mean(axis=1)
            if len(seed_dfs) == 5:
                base_df['Config'] = cfg_label
                print(f"  Ensemble Exp A_TUNED: {cfg_label:<24} averaged across 5 seeds ({len(base_df)} rows)")
            else:
                partial_cfg_name = f"{cfg_label}_partial{len(seed_dfs)}seeds"
                base_df['Config'] = partial_cfg_name
                PARTIAL_ENSEMBLES.append(f"{partial_cfg_name} ({len(seed_dfs)}/5 seeds completed)")
                print(f"  ⚠️ Partial Ensemble Exp A_TUNED: {partial_cfg_name:<30} averaged across {len(seed_dfs)}/5 seeds ({len(base_df)} rows)")
            base_df['Stage'] = 'A_TUNED'
            ensemble_rows.append(base_df)

# 3. Experiment B Ensembles (Concatenating refit blocks)
for m_name, fset_name, loss_choice in exp_b_configs:
    cfg_label = f'{m_name}_{loss_choice}_{fset_name}_refit60d'
    block_dfs = []
    for b_num, _, _ in blocks:
        p_file = os.path.join(OUTPUT_DIR, f'v7_expB_partial_{m_name}_{loss_choice}_{fset_name}_blk{b_num}.csv')
        if os.path.exists(p_file):
            block_dfs.append(pd.read_csv(p_file))
            
    if len(block_dfs) == len(blocks):
        b_full = pd.concat(block_dfs, ignore_index=True)
        b_df = b_full[['cutoff', 'real_date', 'unique_id', 'ds', 'y']].copy()
        b_df['prediction'] = b_full[m_name].values
        b_df['Config'] = cfg_label
        b_df['Stage'] = 'B'
        ensemble_rows.append(b_df)
        print(f"  Ensemble Exp B: {cfg_label:<24} concatenated across {len(block_dfs)} blocks ({len(b_df)} rows)")

if len(ensemble_rows) > 0:
    df_all_ens = pd.concat(ensemble_rows, ignore_index=True)
    ensemble_csv = os.path.join(OUTPUT_DIR, 'v7_ensemble_predictions.csv')
    atomic_save_csv(df_all_ens, ensemble_csv)
    print(f"\n✅ Total ensemble tersimpan ke {ensemble_csv}: {len(df_all_ens)} baris ({df_all_ens['Config'].nunique()} konfigurasi).")
else:
    print("\n⚠️ Belum ada partial run yang selesai untuk diagregasi.")


=== AGREGASI PREDIKSI ENSEMBLE KE FORMAT V6 ===
  Ensemble Exp A_TUNED: NBEATSx_MSEtuned_SPILL   averaged across 5 seeds (3350 rows)
  Ensemble Exp A_TUNED: NBEATSx_MSEtuned_NOSPILL averaged across 5 seeds (3350 rows)
  Ensemble Exp A_TUNED: TFT_MAEtuned_SPILL       averaged across 5 seeds (3350 rows)
  Ensemble Exp A_TUNED: TFT_MAEtuned_NOSPILL     averaged across 5 seeds (3350 rows)
  Ensemble Exp B: TFT_MSE_SPILL_refit60d   concatenated across 6 blocks (3350 rows)
  Ensemble Exp B: TFT_MSE_NOSPILL_refit60d concatenated across 6 blocks (3350 rows)
  Ensemble Exp B: NBEATSx_MAE_SPILL_refit60d concatenated across 6 blocks (3350 rows)
  Ensemble Exp B: NBEATSx_MAE_NOSPILL_refit60d concatenated across 6 blocks (3350 rows)

✅ Total ensemble tersimpan ke /kaggle/working/v7_ensemble_predictions.csv: 26800 baris (8 konfigurasi).


# CELL 10: POST-PROCESSING, METRIK EVALUASI, UJI DM & MODEL CONFIDENCE SET (MCS)


In [10]:
from scipy import stats

def compute_qlike(y_true, y_pred):
    yt = np.array(y_true, dtype=float)
    yp = np.array(y_pred, dtype=float)
    ratio = yt / yp
    return ratio - np.log(ratio) - 1.0

def diebold_mariano_test(loss1, loss2, h=1):
    d = loss1 - loss2
    n = len(d)
    mean_d = np.mean(d)
    gamma0 = np.var(d, ddof=0)
    gamma_sum = 0.0
    for lag in range(1, h):
        gamma_lag = np.mean((d[lag:] - mean_d) * (d[:-lag] - mean_d))
        gamma_sum += 2.0 * gamma_lag
    v_d = (gamma0 + gamma_sum) / n
    if v_d <= 0:
        v_d = 1e-12
    dm_stat = mean_d / np.sqrt(v_d)
    hln_factor = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    hln_stat = dm_stat * hln_factor
    p_val = 2.0 * (1.0 - stats.t.cdf(np.abs(hln_stat), df=n - 1))
    return float(hln_stat), float(p_val)

t0_stage_post = time.time()
ensemble_csv = os.path.join(OUTPUT_DIR, 'v7_ensemble_predictions.csv')

if os.path.exists(ensemble_csv):
    df_ens = pd.read_csv(ensemble_csv)
    df_ens['cutoff'] = df_ens['cutoff'].astype(str)
    df_ens['real_date'] = df_ens['real_date'].astype(str)
    df_ens['h'] = df_ens.groupby(['cutoff', 'Config']).cumcount() + 1
    
    # Discover required baseline files via find_file (searching configured dirs and /kaggle/input)
    candidate_baseline_dirs = ['/kaggle/input/datasets/ferastoro/spillover-v5-inputs', '/kaggle/input/datasets/ferastoro/spillover-v5-inputs/rsic-v5-inputs', DATA_DIR, FALLBACK_INPUT_DIR, os.path.join(BASE_DIR, 'results', 'v6'), os.path.join(BASE_DIR, 'inputs', 'v5')]
    baseline_targets = {
        'har_h1': 'har_preds_T0_h1_v2.csv',
        'har_h10': 'har_preds_T0_h10_v2.csv',
        'garch': 'GARCH_Predictions_V4.csv',
        'v6': 'v6_ensemble_predictions.csv'
    }
    baseline_paths = {}
    
    for b_key, b_fname in baseline_targets.items():
        try:
            p = find_file(b_fname, candidate_baseline_dirs)
            baseline_paths[b_key] = p
            print(f"✅ Found baseline {b_fname:<30}: {p}")
        except FileNotFoundError:
            baseline_paths[b_key] = None
            if b_fname not in MISSING_BASELINES:
                MISSING_BASELINES.append(b_fname)
            print(f"❌ MISSING baseline {b_fname:<30}: Not found!")

    if MISSING_BASELINES:
        print("!" * 88)
        print("CRITICAL ERROR: MISSING BASELINE PREDICTION FILES REQUIRED FOR POST-PROCESSING")
        for mbf in MISSING_BASELINES:
            print(f"  - MISSING BASELINE: {mbf}")
        print("!" * 88)
        
        # In full stages, stop immediately before post-processing starts (no silent df_v6 = None)
        if STAGE in ['A_TUNED', 'B', 'ALL']:
            raise FileNotFoundError(f"Missing required baseline files for post-processing in STAGE {STAGE}: {MISSING_BASELINES}. Ensure all 4 baseline files are uploaded to Kaggle dataset.")

    har_h1 = pd.read_csv(baseline_paths['har_h1']).sort_values('Date').reset_index(drop=True) if baseline_paths['har_h1'] else None
    har_h10 = pd.read_csv(baseline_paths['har_h10']).sort_values('Date').reset_index(drop=True) if baseline_paths['har_h10'] else None
    
    if baseline_paths['garch']:
        garch_df = pd.read_csv(baseline_paths['garch'])
        garch_df['h'] = garch_df.groupby('cutoff').cumcount() + 1
        garch_h1 = garch_df[garch_df['h'] == 1].sort_values('real_date').reset_index(drop=True)
        garch_h10 = garch_df[garch_df['h'] == 10].sort_values('real_date').reset_index(drop=True)
    else:
        garch_h1 = garch_h10 = None
        
    df_v6 = pd.read_csv(baseline_paths['v6']) if baseline_paths['v6'] else None
    if df_v6 is not None:
        df_v6['cutoff'] = df_v6['cutoff'].astype(str)
        df_v6['real_date'] = df_v6['real_date'].astype(str)
        df_v6['h'] = df_v6.groupby(['cutoff', 'Config']).cumcount() + 1
        
    eval_metrics = []
    all_preds_dict = {1: {}, 10: {}}
    
    for h_val in [1, 10]:
        gt_h = har_h1 if h_val == 1 else har_h10
        g_h = garch_h1 if h_val == 1 else garch_h10
        y_true = gt_h['True'].values
        mean_y = float(np.mean(y_true))
        
        bench_preds = {
            'HAR': gt_h['HAR'].values,
            'Hist-Exp': gt_h['Hist_Exp'].values,
            'Roll-20': gt_h['Roll_20'].values,
            'GARCH(1,1)': g_h['GARCH11'].values,
            'EGARCH(1,1)': g_h['EGARCH'].values
        }
        for b_name, b_pred in bench_preds.items():
            all_preds_dict[h_val][b_name] = b_pred
            
        if df_v6 is not None:
            for v6_cfg in df_v6['Config'].unique():
                sub_v6 = df_v6[(df_v6['Config'] == v6_cfg) & (df_v6['h'] == h_val)].sort_values(['cutoff', 'real_date']).reset_index(drop=True)
                if len(sub_v6) == len(y_true):
                    all_preds_dict[h_val][f"v6_{v6_cfg}"] = sub_v6['prediction'].values
                    
        for v7_cfg in df_ens['Config'].unique():
            sub_v7 = df_ens[(df_ens['Config'] == v7_cfg) & (df_ens['h'] == h_val)].sort_values(['cutoff', 'real_date']).reset_index(drop=True)
            if len(sub_v7) == len(y_true):
                all_preds_dict[h_val][v7_cfg] = sub_v7['prediction'].values
                
        hist_exp_mse = np.mean((y_true - all_preds_dict[h_val]['Hist-Exp'])**2)
        
        for m_lbl, m_pred in all_preds_dict[h_val].items():
            mse = float(np.mean((y_true - m_pred)**2))
            mae = float(np.mean(np.abs(y_true - m_pred)))
            r2_oos = float(1.0 - (mse / hist_exp_mse)) if hist_exp_mse > 0 else np.nan
            bias = float(np.mean(m_pred - y_true))
            mean_f = float(np.mean(m_pred))
            ratio_f = mean_f / mean_y if mean_y > 0 else np.nan
            share_2x = float(np.mean(m_pred > (2.0 * mean_y)) * 100.0)
            neg_pct = float(np.mean(m_pred < 0) * 100.0)
            
            outlier_mask = np.isin(gt_h['Date'].astype(str).values, ['2026-02-05', '2026-02-06'])
            se_all = (y_true - m_pred)**2
            se_outliers = np.sum(se_all[outlier_mask])
            outlier_share = float((se_outliers / np.sum(se_all)) * 100.0) if np.sum(se_all) > 0 else np.nan
            mse_no_outliers = float(np.mean(se_all[~outlier_mask]))
            
            loss_lbl = 'MAE' if 'MAE' in m_lbl else ('MSE' if 'MSE' in m_lbl else 'N/A')
            
            eval_metrics.append({
                'Horizon': f"h={h_val}",
                'Model': m_lbl,
                'Training_Loss': loss_lbl,
                'MSE': mse,
                'MAE': mae,
                'R2_OOS_vs_HistExp': r2_oos,
                'Mean_Forecast_Error_Bias': bias,
                'Mean_Forecast': mean_f,
                'Forecast_to_Target_Ratio': ratio_f,
                'Share_Above_2x_Target_Mean (%)': share_2x,
                'Share_Negative_Forecasts (%)': neg_pct,
                'Outlier_Share_Feb5_6 (%)': outlier_share,
                'MSE_Without_Feb5_6': mse_no_outliers
            })
            
    df_eval_summary = pd.DataFrame(eval_metrics)
    summary_csv = os.path.join(OUTPUT_DIR, 'v7_evaluation_metrics_summary.csv')
    atomic_save_csv(df_eval_summary, summary_csv)
    print(f"✅ Saved metrics summary to {summary_csv}")
    
    # ----------------------------------------------------------------------------------
    # QLIKE RECOMPUTATION: Common Positive Subset vs Floored 1e-6 vs Floored 1e-8
    # ----------------------------------------------------------------------------------
    qlike_records = []
    common_masks = {}
    
    for h_val in [1, 10]:
        y_true = har_h1['True'].values if h_val == 1 else har_h10['True'].values
        pos_mask = (y_true > 0)
        for m_lbl, m_pred in all_preds_dict[h_val].items():
            pos_mask = pos_mask & (m_pred > 0)
        common_masks[h_val] = pos_mask
        n_common = int(np.sum(pos_mask))
        y_com = y_true[pos_mask]
        
        for m_lbl, m_pred in all_preds_dict[h_val].items():
            q_com = float(np.mean(compute_qlike(y_com, m_pred[pos_mask]))) if n_common > 0 else np.nan
            q_f6 = float(np.mean(compute_qlike(y_true, np.maximum(m_pred, 1e-6))))
            q_f8 = float(np.mean(compute_qlike(y_true, np.maximum(m_pred, 1e-8))))
            
            qlike_records.append({
                'Horizon': f"h={h_val}",
                'Model': m_lbl,
                'N_Common_Positive': n_common,
                'QLIKE_Common_Positive': q_com,
                'N_Full_Sample': len(y_true),
                'QLIKE_Floored_1e-6 (Exploratory)': q_f6,
                'QLIKE_Floored_1e-8 (Exploratory)': q_f8
            })
            
    df_qlike = pd.DataFrame(qlike_records)
    qlike_csv = os.path.join(OUTPUT_DIR, 'v7_qlike_rules_comparison.csv')
    atomic_save_csv(df_qlike, qlike_csv)
    print(f"✅ Saved QLIKE comparison to {qlike_csv} (Common positive N={sum(common_masks[1])} at h=1, N={sum(common_masks[10])} at h=10)")
    
    # ----------------------------------------------------------------------------------
    # DIEBOLD-MARIANO TESTS (with MSE ratio vs HAR, raw p, Bonferroni m stated)
    # ----------------------------------------------------------------------------------
    dm_records = []
    for h_val in [1, 10]:
        y_true = har_h1['True'].values if h_val == 1 else har_h10['True'].values
        har_pred = all_preds_dict[h_val]['HAR']
        har_mse = np.mean((y_true - har_pred)**2)
        loss_har = (y_true - har_pred)**2
        
        for m_lbl, m_pred in all_preds_dict[h_val].items():
            if m_lbl == 'HAR':
                continue
            loss_m = (y_true - m_pred)**2
            stat, p_raw = diebold_mariano_test(loss_m, loss_har, h=h_val)
            m_mse = np.mean(loss_m)
            mse_ratio = float(m_mse / har_mse)
            m_total = len(all_preds_dict[h_val]) - 1
            
            dm_records.append({
                'Horizon': f"h={h_val}",
                'Candidate': m_lbl,
                'Benchmark': 'HAR',
                'MSE_Ratio_vs_HAR': mse_ratio,
                'Pct_Difference_vs_HAR (%)': (mse_ratio - 1.0) * 100.0,
                'DM_Stat': stat,
                'p_raw': p_raw,
                'm_Bonferroni': m_total,
                'p_bonf': min(p_raw * m_total, 1.0),
                'DM_Verdict': 'better' if stat < 0 and p_raw < 0.05 else ('worse' if stat > 0 and p_raw < 0.05 else 'tie')
            })
            
    df_dm = pd.DataFrame(dm_records)
    dm_csv = os.path.join(OUTPUT_DIR, 'v7_diebold_mariano_tests.csv')
    atomic_save_csv(df_dm, dm_csv)
    print(f"✅ Saved Diebold-Mariano tests to {dm_csv}")
    
    # ----------------------------------------------------------------------------------
    # MODEL CONFIDENCE SET (MCS) on MSE and QLIKE
    # ----------------------------------------------------------------------------------
    try:
        from arch.bootstrap import MCS
        mcs_records = []
        for h_val in [1, 10]:
            y_true = har_h1['True'].values if h_val == 1 else har_h10['True'].values
            mse_dict = {m: (y_true - p)**2 for m, p in all_preds_dict[h_val].items()}
            mcs_mse = MCS(pd.DataFrame(mse_dict), size=0.10, reps=5000, seed=42)
            mcs_mse.compute()
            pvals_mse = mcs_mse.pvalues.iloc[:, 0].to_dict()
            
            pos_m = common_masks[h_val]
            y_c = y_true[pos_m]
            if len(y_c) > 10:
                qlike_dict = {m: compute_qlike(y_c, p[pos_m]) for m, p in all_preds_dict[h_val].items()}
                mcs_q = MCS(pd.DataFrame(qlike_dict), size=0.10, reps=5000, seed=42)
                mcs_q.compute()
                pvals_q = mcs_q.pvalues.iloc[:, 0].to_dict()
            else:
                pvals_q = {}
                
            for m in all_preds_dict[h_val]:
                mcs_records.append({
                    'Horizon': f"h={h_val}",
                    'Model': m,
                    'MCS_pvalue_MSE': pvals_mse.get(m, np.nan),
                    'In_M90_MSE': pvals_mse.get(m, 0.0) >= 0.10,
                    'MCS_pvalue_QLIKE': pvals_q.get(m, np.nan),
                    'In_M90_QLIKE': pvals_q.get(m, 0.0) >= 0.10
                })
        df_mcs = pd.DataFrame(mcs_records)
        mcs_csv = os.path.join(OUTPUT_DIR, 'v7_mcs_results.csv')
        atomic_save_csv(df_mcs, mcs_csv)
        print(f"✅ Saved MCS results to {mcs_csv}")
    except Exception as e:
        print(f"Notice: MCS computation skipped ({e})")
    STAGE_TIMINGS['POSTPROCESSING'] = time.time() - t0_stage_post


✅ Found baseline har_preds_T0_h1_v2.csv        : /kaggle/input/datasets/ferastoro/spillover-v5-inputs/rsic-v5-inputs/har_preds_T0_h1_v2.csv
✅ Found baseline har_preds_T0_h10_v2.csv       : /kaggle/input/datasets/ferastoro/spillover-v5-inputs/rsic-v5-inputs/har_preds_T0_h10_v2.csv
✅ Found baseline GARCH_Predictions_V4.csv      : /kaggle/input/datasets/ferastoro/spillover-v5-inputs/rsic-v5-inputs/GARCH_Predictions_V4.csv
✅ Found baseline v6_ensemble_predictions.csv   : /kaggle/input/datasets/ferastoro/spillover-v5-inputs/v6_ensemble_predictions.csv
✅ Saved metrics summary to /kaggle/working/v7_evaluation_metrics_summary.csv
✅ Saved QLIKE comparison to /kaggle/working/v7_qlike_rules_comparison.csv (Common positive N=169 at h=1, N=182 at h=10)
✅ Saved Diebold-Mariano tests to /kaggle/working/v7_diebold_mariano_tests.csv
✅ Saved MCS results to /kaggle/working/v7_mcs_results.csv


# CELL 11: MANIFEST GENERATION, README DOKUMENTASI & ZIP PACKAGING


In [11]:
print('=== MEMBUAT DOKUMENTASI README, MANIFEST MD5 & ZIP PACKAGE ===')
import shutil
import zipfile

run_log_file = os.path.join(OUTPUT_DIR, 'run_log.csv')
finished_runs = []
if os.path.exists(run_log_file):
    try:
        df_rl = pd.read_csv(run_log_file)
        finished_runs = df_rl['run_id'].tolist()
    except Exception:
        finished_runs = []

expected_runs = []
if STAGE in ['A', 'SMOKE']:
    for m, f, l in exp_a_configs:
        for s in exp_a_seeds:
            expected_runs.append(f"A_{m}_{l}_{f}_s{s}")
if STAGE in ['A_TUNED', 'ALL', 'SMOKE']:
    if 'tuned_configs' in locals():
        for m, f, l, _ in tuned_configs:
            for s in tuned_seeds:
                expected_runs.append(f"A_TUNED_{l}_s{s}")
if STAGE in ['B', 'ALL', 'SMOKE']:
    for m, f, l in exp_b_configs:
        for b_num, _, _ in blocks:
            expected_runs.append(f"B_{m}_{l}_{f}_blk{b_num}")

missing_runs = [r for r in expected_runs if r not in finished_runs]

avg_fit_sec = 0.0
if os.path.exists(run_log_file):
    try:
        df_rl = pd.read_csv(run_log_file)
        if 'fit_seconds' in df_rl.columns and len(df_rl) > 0:
            avg_fit_sec = float(df_rl['fit_seconds'].mean())
    except Exception:
        avg_fit_sec = 0.0

smoke_estimates_text = ""
if STAGE == 'SMOKE':
    # Updated runtime estimation:
    # A_TUNED: (12 NBEATSx combos * 2 seeds + 8 TFT combos * 2 seeds = 40 tuning fits) + (20 winner 5-seed fits) = 60 fits
    # Stage B: 4 configs * 6 blocks = 24 fits
    est_atuned_hours = (40 * avg_fit_sec * 20 + 20 * avg_fit_sec * 20) / 3600.0 if avg_fit_sec > 0 else 0.6
    est_b_hours = (24 * avg_fit_sec * 20) / 3600.0 if avg_fit_sec > 0 else 0.4
    smoke_estimates_text = f"""
Runtime Estimates from SMOKE (T4 GPU Extrapolation for New Grids):
  - Measured Time per Fit (SMOKE): {avg_fit_sec:.2f} seconds
  - Estimated Total Runtime for STAGE A_TUNED (60 fits total): ~{est_atuned_hours:.2f} hours
  - Estimated Total Runtime for STAGE B       (24 fits total): ~{est_b_hours:.2f} hours
  - Estimated Total Runtime for STAGE ALL     (84 fits total): ~{est_atuned_hours + est_b_hours:.2f} hours
"""

# 1. Search and copy notebook source into output directory (Fix 1)
notebook_copy_path = os.path.join(OUTPUT_DIR, 'notebook_source.ipynb')
candidate_notebooks = [
    os.path.abspath('08_dl_v7_audit_loss_refit.ipynb'),
    os.path.abspath('notebooks/08_dl_v7_audit_loss_refit.ipynb'),
    os.path.abspath('../08_dl_v7_audit_loss_refit.ipynb'),
    os.path.abspath('__notebook_source__.ipynb'),
    os.path.abspath('.virtual_documents/__notebook_source__.ipynb'),
    '/kaggle/working/__notebook_source__.ipynb',
    '/kaggle/working/.virtual_documents/__notebook_source__.ipynb'
]
nb_source_found = False
nb_source_file_used = None
for c_nb in candidate_notebooks:
    if os.path.exists(c_nb):
        shutil.copy2(c_nb, notebook_copy_path)
        nb_source_found = True
        nb_source_file_used = c_nb
        break

if nb_source_found:
    notebook_status_readme = f"Source Notebook Copy : Saved to notebook_source.ipynb (copied from {nb_source_file_used})"
    print(f"✅ Salinan notebook sumber disimpan: {notebook_copy_path} (from {nb_source_file_used})")
else:
    missing_txt_path = os.path.join(OUTPUT_DIR, 'notebook_source_MISSING.txt')
    with open(missing_txt_path, 'w') as f_miss:
        f_miss.write("WARNING: Notebook source file not found during packaging.\nChecked paths:\n" + "\n".join(['  - ' + p for p in candidate_notebooks]) + "\n")
    notebook_status_readme = "Source Notebook Copy : NOT FOUND! Generated notebook_source_MISSING.txt"
    print(f"⚠️ Notebook source tidak ditemukan! Dibuat: {missing_txt_path}")

# Format Stage Timings (Fix 6)
timing_lines = []
for stg_k in ['A', 'A_TUNED', 'B', 'POSTPROCESSING']:
    stg_t = STAGE_TIMINGS.get(stg_k)
    if stg_t is not None:
        timing_lines.append(f"  - Stage {stg_k:<15} : {stg_t:.2f} seconds ({stg_t/60.0:.2f} minutes)")
    else:
        timing_lines.append(f"  - Stage {stg_k:<15} : Not Run")
timing_text = "\n".join(timing_lines)

# Edge check summary from tuning
edge_info_lines = []
if 'winner_edge_notes' in locals():
    for m_k, e_v in winner_edge_notes.items():
        edge_info_lines.append(f"  - {m_k:<8} Winner Edge Status : {e_v}")
edge_info_text = "\n".join(edge_info_lines) if edge_info_lines else "  - (Tuning edge check not executed in this run)"

grid_doc_text = f"""Hyperparameter Tuning Grid Specifications & Settings:
  - v7 NBEATSx Grid (12 combos) : input_size in {{20, 50, 100}} x learning_rate in {{1e-3, 5e-4, 3e-4, 1e-4}} | MSE loss only
  - v7 TFT Grid (8 combos)      : input_size in {{20, 50}} x learning_rate in {{1e-3, 7.5e-4, 5e-4, 3e-4}} | MAE loss only
  - Tuning Ranking Metric       : Mean validation MSE over seeds 42 and 7 (averaged over h=1 and h=10)
  
Early Stopping & Verification Settings:
  - NBEATSx Tuning & Winner : val_check_steps = 5 (v6 used 25) | patience = 25 checks (v6 used 5)
                              Total patience in steps = 125 steps (5 * 25 = 125, identical to v6: 25 * 5 = 125).
                              Restore-best weights: Enabled.
  - TFT Tuning & Winner     : val_check_steps = 25 | patience = 5 checks (125 steps). Restore-best: Enabled.

Grid Edge Verification:
{edge_info_text}

v6 Stage B Empirical Tuning Results for NBEATSx (from results/v6/v6_tuning_log.csv):
  +--------------------+------------+--------------+------------------+------------------+
  | Hyperparameters    | input_size | lr           | MAE Train ValMSE | MSE Train ValMSE |
  +--------------------+------------+--------------+------------------+------------------+
  | Combo 1 (cfg1/2)   | 20         | 0.0010       | 1.774417e-06     | 9.471862e-06     |
  | Combo 2 (cfg3/4)*  | 20         | 0.0003       | 1.772739e-06 (*) | 5.141590e-06     |
  | Combo 3 (cfg5/6)   | 50         | 0.0010       | 2.338505e-06     | 3.573679e-06     |
  | Combo 4 (cfg7/8)   | 50         | 0.0003       | 2.277654e-06     | 2.836472e-06     |
  +--------------------+------------+--------------+------------------+------------------+
  (*) Winner in v6 Stage B: NBEATSx_cfg3 (MAE loss, input 20, lr 0.0003)."""

# Missing Baselines & Partial Ensembles sections (Fix 2 & Fix 3)
missing_baselines_text = ""
if MISSING_BASELINES:
    missing_baselines_text = "\nMissing Baseline Files for Post-Processing:\n" + "\n".join([f"  - MISSING BASELINE: {b}" for b in MISSING_BASELINES]) + "\n"

partial_ensembles_text = ""
if PARTIAL_ENSEMBLES:
    partial_ensembles_text = "\nPartial Seed Ensembles (<5 seeds):\n" + "\n".join([f"  - {pe}" for pe in PARTIAL_ENSEMBLES]) + "\n"

# 2. WRITE README.TXT FIRST (Before Manifest)
readme_path = os.path.join(OUTPUT_DIR, 'README.txt')
readme_content = f"""================================================================================
RSIC Volatility Forecasting Audit (v7)
Loss-Matched Retraining & 60-Day Periodic Refitting
================================================================================
Execution Timestamp  : {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}
Active STAGE         : {STAGE}
Hardware Device      : {gpu_name} (CUDA={gpu_available})
NeuralForecast Ver   : {nf_ver}
Evaluated Seeds      : {SEEDS_BANNER}
Test Cutoffs Target  : {N_TEST_CUTOFFS} cutoffs
Runtime Budget Limit : {MAX_RUNTIME_HOURS} hours
Total Elapsed Time   : {(time.time() - T_GLOBAL_START)/3600:.2f} hours
{notebook_status_readme}

Actual Stage Elapsed Times:
{timing_text}

{grid_doc_text}
{missing_baselines_text}{partial_ensembles_text}
Run Completion Summary:
  - Total Expected Runs : {len(expected_runs)}
  - Finished Runs Count : {len(finished_runs)}
  - Missing Runs Count  : {len(missing_runs)}

Finished Runs:
{chr(10).join(['  + ' + r for r in finished_runs]) if finished_runs else '  (None)'}

Missing Runs:
{chr(10).join(['  - ' + r for r in missing_runs]) if missing_runs else '  (None - All expected runs completed successfully)'}

Measured Fit Timing:
  - Average Fit Time : {avg_fit_sec:.2f} seconds per fit
{smoke_estimates_text}
================================================================================
"""
with open(readme_path, 'w') as f_rm:
    f_rm.write(readme_content)
print(f"✅ README.txt ditulis pertama: {readme_path}")

# 3. GENERATE MANIFEST.CSV (AFTER README.txt and notebook_source.ipynb)
manifest_files = []
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in sorted(files):
        fpath = os.path.join(root, f)
        rel_path = os.path.relpath(fpath, OUTPUT_DIR)
        if f not in ['manifest.csv', 'v7_results.zip']:
            manifest_files.append({
                'File': rel_path,
                'Size_Bytes': os.path.getsize(fpath),
                'MD5': get_file_md5(fpath)
            })

df_manifest = pd.DataFrame(manifest_files)
manifest_csv = os.path.join(OUTPUT_DIR, 'manifest.csv')
atomic_save_csv(df_manifest, manifest_csv)
print(f'✅ Manifest MD5 tersimpan: {manifest_csv} ({len(df_manifest)} files terverifikasi)')

# 4. ZIP ALL CONTENTS INTO v7_results.zip
zip_path = os.path.join(OUTPUT_DIR, 'v7_results.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for f in sorted(files):
            if f != 'v7_results.zip':
                fpath = os.path.join(root, f)
                rel_path = os.path.relpath(fpath, OUTPUT_DIR)
                zipf.write(fpath, arcname=rel_path)
print(f'✅ Seluruh hasil berhasil dipaketkan ke zip: {zip_path} ({os.path.getsize(zip_path)} bytes)')
print('🎉 Semua tahapan pipeline selesai dengan sukses!')


=== MEMBUAT DOKUMENTASI README, MANIFEST MD5 & ZIP PACKAGE ===
⚠️ Notebook source tidak ditemukan! Dibuat: /kaggle/working/notebook_source_MISSING.txt
✅ README.txt ditulis pertama: /kaggle/working/README.txt
✅ Manifest MD5 tersimpan: /kaggle/working/manifest.csv (139 files terverifikasi)
✅ Seluruh hasil berhasil dipaketkan ke zip: /kaggle/working/v7_results.zip (1319195 bytes)
🎉 Semua tahapan pipeline selesai dengan sukses!
